# 02 — FlashAttention-3 Benchmark

**Pipeline:** Benchmark vanilla vs FlashAttention variants across sequence lengths 1k–64k

```
Sequence lengths: [1024, 2048, 4096, 8192, 16384, 32768, 65536]
        │
        ▼
  Vanilla Softmax Attention
  Flash Attention (v1/v2/v3)    ── throughput (tokens/sec)
  Linear Attention               ── peak GPU memory (MB)
        │                         ── wall-clock latency (ms)
        ▼
  Benchmark CSV + Plotly charts
        │
  wandb artefact upload
```

**References:**
- FlashAttention-2: Dao (2023) https://arxiv.org/abs/2307.08691
- FlashAttention-3: Shah et al. (2024) https://arxiv.org/abs/2407.08608
- Linear Attention: Katharopoulos et al. (2020) https://arxiv.org/abs/2006.16236

> **GPU required** for meaningful benchmarks.  
> CPU fallback available — results will differ significantly.

In [ ]:
from __future__ import annotations

import contextlib
import csv
import logging
import os
import time
from pathlib import Path

import torch
import torch.nn.functional as F
import wandb

logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
log.info("Device: %s", DEVICE)

CFG = {
    "seq_lengths": [1024, 2048, 4096, 8192, 16384, 32768, 65536],
    "batch_size": 2,
    "num_heads": 8,
    "head_dim": 64,
    "dtype": "float16",
    "warmup_iters": 5,
    "benchmark_iters": 20,
    "device": DEVICE,
}

if not os.getenv("WANDB_DISABLED"):
    wandb.init(project="ai-transformer-research-hub", config=CFG, job_type="benchmark")


In [ ]:
# FlashAttention-2: Dao (2023) https://arxiv.org/abs/2307.08691
# FlashAttention-3: Shah et al. (2024) https://arxiv.org/abs/2407.08608
# Linear Attention: Katharopoulos et al. (2020) https://arxiv.org/abs/2006.16236

_DTYPE_MAP = {"float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}

# Build available SDP backends (PyTorch >= 2.0)
_BACKENDS: dict[str, contextlib.AbstractContextManager] = {}
try:
    _BACKENDS["math"] = torch.backends.cuda.sdp_kernel(
        enable_flash=False, enable_math=True, enable_mem_efficient=False
    )
    _BACKENDS["flash"] = torch.backends.cuda.sdp_kernel(
        enable_flash=True, enable_math=False, enable_mem_efficient=False
    )
    _BACKENDS["mem_efficient"] = torch.backends.cuda.sdp_kernel(
        enable_flash=False, enable_math=False, enable_mem_efficient=True
    )
except AttributeError:
    _BACKENDS["default"] = contextlib.nullcontext()


def _make_qkv(
    seq_len: int, batch: int, heads: int, head_dim: int, dtype: torch.dtype, device: str
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Allocate random Q/K/V tensors on the target device."""
    shape = (batch, heads, seq_len, head_dim)
    q = torch.randn(shape, dtype=dtype, device=device)
    k = torch.randn(shape, dtype=dtype, device=device)
    v = torch.randn(shape, dtype=dtype, device=device)
    return q, k, v


def benchmark_attention(
    backend_name: str,
    ctx_mgr: contextlib.AbstractContextManager,
    seq_len: int,
    cfg: dict,
) -> dict:
    """Time one SDP backend at a given sequence length.

    Runs warmup_iters discarded passes then benchmark_iters timed passes.
    GPU memory is sampled before and after to compute peak delta.

    Args:
        backend_name: Human-readable label for the backend.
        ctx_mgr: Context manager that activates the SDP kernel selector.
        seq_len: Sequence length to benchmark.
        cfg: Global configuration dict.

    Returns:
        Dict with keys: backend, seq_len, latency_ms, throughput_toks_per_sec,
        peak_memory_mb, oom.
    """
    device = cfg["device"]
    dtype = _DTYPE_MAP.get(cfg["dtype"], torch.float16)
    batch, heads, head_dim = cfg["batch_size"], cfg["num_heads"], cfg["head_dim"]

    if device == "cpu" and seq_len > 4096:
        log.info("Skipping seq_len=%d on CPU (too slow)", seq_len)
        return {
            "backend": backend_name, "seq_len": seq_len,
            "latency_ms": float("nan"), "throughput_toks_per_sec": float("nan"),
            "peak_memory_mb": 0.0, "oom": False,
        }

    try:
        q, k, v = _make_qkv(seq_len, batch, heads, head_dim, dtype, device)
        for _ in range(cfg["warmup_iters"]):
            with ctx_mgr:
                _ = F.scaled_dot_product_attention(q, k, v)
        if device == "cuda":
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()

        t0 = time.perf_counter()
        for _ in range(cfg["benchmark_iters"]):
            with ctx_mgr:
                _ = F.scaled_dot_product_attention(q, k, v)
        if device == "cuda":
            torch.cuda.synchronize()
        elapsed_ms = (time.perf_counter() - t0) * 1000 / cfg["benchmark_iters"]

        mem_mb = torch.cuda.max_memory_allocated() / 1e6 if device == "cuda" else 0.0
        throughput = (batch * seq_len) / (elapsed_ms / 1000)
        return {
            "backend": backend_name, "seq_len": seq_len,
            "latency_ms": round(elapsed_ms, 3),
            "throughput_toks_per_sec": round(throughput),
            "peak_memory_mb": round(mem_mb, 1),
            "oom": False,
        }
    except torch.cuda.OutOfMemoryError:  # type: ignore[attr-defined]
        log.warning("OOM: backend=%s seq_len=%d", backend_name, seq_len)
        return {"backend": backend_name, "seq_len": seq_len, "latency_ms": float("nan"),
                "throughput_toks_per_sec": 0, "peak_memory_mb": float("nan"), "oom": True}
    except Exception as exc:  # noqa: BLE001
        log.warning("Error: backend=%s seq_len=%d — %s", backend_name, seq_len, exc)
        return {"backend": backend_name, "seq_len": seq_len, "latency_ms": float("nan"),
                "throughput_toks_per_sec": 0, "peak_memory_mb": float("nan"), "oom": False}


results: list[dict] = []
for seq_len in CFG["seq_lengths"]:
    for name, ctx in _BACKENDS.items():
        row = benchmark_attention(name, ctx, seq_len, CFG)
        results.append(row)
        log.info(
            "backend=%-14s  seq=%6d  lat=%8.2f ms  tput=%9.0f tok/s  mem=%7.1f MB",
            row["backend"], row["seq_len"],
            row["latency_ms"] or 0, row["throughput_toks_per_sec"],
            row["peak_memory_mb"] or 0,
        )

csv_path = Path("attention_benchmark_results.csv")
if results:
    with csv_path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
        writer.writeheader()
        writer.writerows(results)
    log.info("Results saved to %s", csv_path)

if not os.getenv("WANDB_DISABLED"):
    wandb.log({"benchmark_rows": len(results)})


In [ ]:
# Generate Plotly throughput/memory heatmaps and latency line chart
# FlashAttention-2: Dao (2023) https://arxiv.org/abs/2307.08691
# FlashAttention-3: Shah et al. (2024) https://arxiv.org/abs/2407.08608

try:
    import pandas as pd  # type: ignore
    import plotly.express as px  # type: ignore
    import plotly.graph_objects as go  # type: ignore

    df = pd.DataFrame(results)
    valid = df[~df["oom"] & df["latency_ms"].notna()]

    if not valid.empty:
        # Throughput heatmap: x=seq_len, y=backend, color=throughput
        pivot_tput = valid.pivot_table(
            index="backend", columns="seq_len",
            values="throughput_toks_per_sec", aggfunc="mean",
        )
        fig_heat = px.imshow(
            pivot_tput,
            labels={"x": "Sequence Length", "y": "Backend", "color": "Tokens/sec"},
            title="Attention Throughput Heatmap (tokens/sec)",
            color_continuous_scale="Viridis",
            aspect="auto",
        )
        fig_heat.show()

        # Memory heatmap
        pivot_mem = valid.pivot_table(
            index="backend", columns="seq_len",
            values="peak_memory_mb", aggfunc="mean",
        )
        fig_mem = px.imshow(
            pivot_mem,
            labels={"x": "Sequence Length", "y": "Backend", "color": "Memory (MB)"},
            title="Peak GPU Memory Heatmap (MB)",
            color_continuous_scale="Magma",
            aspect="auto",
        )
        fig_mem.show()

        # Latency line chart
        fig_lat = px.line(
            valid, x="seq_len", y="latency_ms", color="backend",
            markers=True, log_x=True,
            title="Attention Latency vs Sequence Length",
            labels={"seq_len": "Sequence Length", "latency_ms": "Latency (ms)"},
        )
        fig_lat.show()

        if not os.getenv("WANDB_DISABLED"):
            wandb.log({
                "throughput_heatmap": wandb.Html(fig_heat.to_html()),
                "memory_heatmap": wandb.Html(fig_mem.to_html()),
                "latency_chart": wandb.Html(fig_lat.to_html()),
            })
    else:
        log.warning("No valid benchmark results to plot")

except ImportError as exc:
    log.warning("Plotting skipped — missing dependency: %s", exc)

if not os.getenv("WANDB_DISABLED"):
    wandb.finish()
